# Unified Training and Testing Pipeline (Colab)

This notebook orchestrates the entire workflow for the UDC-SIT project on Google Colab. It is designed to be run step-by-step.

In [ ]:
# --- Configuration ---
import os
from getpass import getpass

# GitHub Details
GIT_USERNAME = "navdeepsingh1609"
REPO_NAME = "CSC2529-Fall-Project"
REPO_URL = f"https://github.com/{GIT_USERNAME}/{REPO_NAME}.git"

# Google Drive Paths
DRIVE_MOUNT_PATH = "/content/drive"
DRIVE_PROJECT_ROOT = f"{DRIVE_MOUNT_PATH}/MyDrive/Computational Imaging Project"
DRIVE_DATASET_PATH = f"{DRIVE_PROJECT_ROOT}/Datasets/UDC-SIT/UDC-SIT"

# Local Colab Paths
COLAB_DATASET_LINK = "/content/dataset/UDC-SIT"
SUBSET_DIR = "data/UDC-SIT_subset"

print("Configuration loaded.")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
print("Mounting Drive...")
drive.mount(DRIVE_MOUNT_PATH)
print("Drive mounted.")

## 2. Clone Repository & Install Dependencies

In [ ]:
%cd /content

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_NAME}...")
    try:
        !git clone {REPO_URL}
    except:
        print("Public clone failed. Using token...")
        token = getpass('Enter GitHub Token: ')
        auth_url = f"https://{token}@github.com/{GIT_USERNAME}/{REPO_NAME}.git"
        !git clone {auth_url}
else:
    print("Repo already exists. Pulling latest...")
    %cd {REPO_NAME}
    !git pull
    %cd ..

%cd {REPO_NAME}

# Initialize MambaIR Submodule
print("Initializing submodules...")
!git submodule update --init --recursive --remote

# Install Requirements
print("Installing dependencies...")
!pip install -q -r requirements.txt
print("Dependencies installed.")

## 3. Data Preparation
Copy dataset tarballs from Drive to local SSD and extract them for faster I/O. Link the testing set.

In [ ]:
# Copy dataset tarballs to local SSD
print("Copying dataset tarballs...")
!mkdir -p /content/dataset
!cp "{DRIVE_PROJECT_ROOT}/Datasets/UDC-SIT/"*.tar.gz /content/dataset/

# Extract training/validation
print("Extracting dataset...")
%cd /content/dataset
!cat *.tar.gz | tar -xzvf -
%cd /content/CSC2529-Fall-Project

# Link Testing Set (if separate)
print("Linking testing set...")
# Assuming testing set is not in the tarballs or we want to link it explicitly
if not os.path.exists("/content/dataset/UDC-SIT/testing"):
    !ln -s "{DRIVE_DATASET_PATH}/testing" "/content/dataset/UDC-SIT/testing"

print("Data preparation complete.")

In [ ]:
print("Creating subset...")
# Uses the refactored scripts/create_subset.py with arguments
!python scripts/create_subset.py \
    --source-root "/content/dataset/UDC-SIT" \
    --target-root "{SUBSET_DIR}" \
    --n-train 30 \
    --n-val 8

print("Subset created.")
!ls -R {SUBSET_DIR} | head -n 10

## 4. Teacher Training
Train the teacher model. Use `v1` or `v2`.

In [ ]:
# Config
MODEL_VARIANT = "v2"   # 'v1' or 'v2'
PRESET = "quick"       # 'quick' or 'full'
DATA_ROOT = SUBSET_DIR if PRESET == "quick" else COLAB_DATASET_LINK
CHECKPOINT_DIR = f"{DRIVE_PROJECT_ROOT}/checkpoints_{PRESET}"

print(f"Training Teacher ({MODEL_VARIANT}) - Preset: {PRESET}")
print(f"Data: {DATA_ROOT}")
print(f"Output: {CHECKPOINT_DIR}")

!python train_teacher.py \
    --model-variant {MODEL_VARIANT} \
    --data-root "{DATA_ROOT}" \
    --preset {PRESET} \
    --drive-checkpoint-dir "{CHECKPOINT_DIR}" \
    --save-loss-history

## 5. Student Training (Knowledge Distillation)
Train the student using the teacher's weights.

In [ ]:
import glob

# Find latest teacher checkpoint
teacher_ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/teacher_{MODEL_VARIANT}_{PRESET}_*.pth"))
if teacher_ckpts:
    TEACHER_WEIGHTS = teacher_ckpts[-1]
    print(f"Using Teacher Checkpoint: {TEACHER_WEIGHTS}")
    
    !python train_student_kd.py \
        --model-variant {MODEL_VARIANT} \
        --data-root "{DATA_ROOT}" \
        --preset {PRESET} \
        --teacher-weights "{TEACHER_WEIGHTS}" \
        --drive-checkpoint-dir "{CHECKPOINT_DIR}" \
        --save-loss-history
else:
    print("!!! No teacher checkpoint found. Cannot train student.")

## 6. Evaluation
Evaluate both Teacher and Student models to generate predictions for comparison.

In [ ]:
RESULTS_DIR = f"{DRIVE_PROJECT_ROOT}/results_{PRESET}"
SPLIT = "val" if PRESET == "quick" else "validation"

# --- 1. Evaluate Teacher ---
teacher_ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/teacher_{MODEL_VARIANT}_{PRESET}_*.pth"))
if teacher_ckpts:
    TEACHER_CKPT = teacher_ckpts[-1]
    print(f"Evaluating Teacher Checkpoint: {TEACHER_CKPT}")
    
    !python testing_udc.py \
        --model-type teacher \
        --model-variant {MODEL_VARIANT} \
        --data-root "{DATA_ROOT}" \
        --split {SPLIT} \
        --checkpoint-path "{TEACHER_CKPT}" \
        --drive-results-root "{RESULTS_DIR}"
else:
    print("!!! No teacher checkpoint found to evaluate.")

# --- 2. Evaluate Student ---
student_ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/student_{MODEL_VARIANT}_{PRESET}_*.pth"))
if student_ckpts:
    STUDENT_CKPT = student_ckpts[-1]
    print(f"Evaluating Student Checkpoint: {STUDENT_CKPT}")
    
    !python testing_udc.py \
        --model-type student \
        --model-variant {MODEL_VARIANT} \
        --data-root "{DATA_ROOT}" \
        --split {SPLIT} \
        --checkpoint-path "{STUDENT_CKPT}" \
        --drive-results-root "{RESULTS_DIR}"
else:
    print("!!! No student checkpoint found to evaluate.")

## 7. Visualization
Generate visualization panels comparing Input, Teacher, Student, and GT.

In [ ]:
VIZ_RESULTS_ROOT = f"{DRIVE_PROJECT_ROOT}/results_srgb_viz"

# Construct paths to predictions based on previous steps
# testing_udc.py saves to {RESULTS_DIR}/{model_name}_{split}
TEACHER_PRED_DIR = f"{RESULTS_DIR}/teacher_{MODEL_VARIANT}_{SPLIT}/npy"
STUDENT_PRED_DIR = f"{RESULTS_DIR}/student_{MODEL_VARIANT}_{SPLIT}/npy"

print("Generating Visualizations...")
print(f"Teacher Preds: {TEACHER_PRED_DIR}")
print(f"Student Preds: {STUDENT_PRED_DIR}")

!python viz_srgb_udc.py \
    --data-root "{DATA_ROOT}" \
    --split {SPLIT} \
    --teacher-pred-dir "{TEACHER_PRED_DIR}" \
    --student-pred-dir "{STUDENT_PRED_DIR}" \
    --results-root "results_srgb_viz" \
    --drive-results-root "{VIZ_RESULTS_ROOT}" \
    --wb-mode none \
    --gamma 1.0